# Reproducing the paper's figures and tables

Run all cells to validate the saved inputs and regenerate the computational figures and table values. The analysis uses local data and does not require model weights or a cluster connection.

`docs/paper-assets.csv` maps manuscript labels to reference files and generated outputs. The manuscript inventory and shipment figures use the **5–95% path bands** produced in the "Inventory and shipments: manuscript path-band figures" section. Mean-confidence plots, 1–99% bands, the heatmap and additional coverage summaries are diagnostics.

Outputs are saved under `results/paper_release_v1/exports/`. Plotting code is included in the cells so that figure styles can be adjusted directly.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display, Markdown
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "code/paper_analysis").is_dir() and (p / "results/paper_release_v1").is_dir()), None)
if root is None:
    raise RuntimeError("Run this notebook from inside the LiberiaRBM-public directory.")
sys.path.insert(0, str(root / "code"))
from paper_analysis.data import RELEASE, SOURCES, BASELINE, POLICIES, read
from paper_analysis.style import load_style
from paper_analysis.validation import validate_inputs
from paper_analysis import tables
from paper_analysis.coverage import load_coverage, VARIABLES

STYLE = load_style()
STYLE.update(font_size=10, ci_alpha=0.18, figsize=(6.4, 3.7))
STYLE["baseline_policies"] = ["E3_mean_rollout", "Scaled_Static"]
# Example: STYLE["table_precision_overrides"] = {"tab:baseline_compact_comparison": {"coverage": 3}}
plt.rcParams.update({
    "font.family": STYLE["font_family"], "font.size": STYLE["font_size"],
    "axes.titlesize": 11, "axes.labelsize": 10, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.fontsize": 9, "lines.linewidth": STYLE["line_width"],
    "axes.spines.top": False, "axes.spines.right": False,
    "pdf.fonttype": 42, "ps.fonttype": 42, "figure.dpi": 110,
})
OUTPUT = RELEASE / "exports/notebook_v2"
for folder in ["figures", "tables", "data"]:
    (OUTPUT / folder).mkdir(parents=True, exist_ok=True)
FIGURE_EXPORTS = {}
# Save the figure; plotting is defined in the cells below.
def save_figure(fig, name, sources, meaning):
    # Freeze the resolved layout so PDF/PNG backends cannot shift or crop labels.
    fig.canvas.draw()
    fig.set_layout_engine("none")
    paths = []
    for extension in ["pdf", "png"]:
        path = OUTPUT / "figures" / f"{name}.{extension}"
        fig.savefig(path, dpi=STYLE["dpi"], bbox_inches="tight", facecolor="white")
        paths.append(str(path.relative_to(RELEASE)))
    FIGURE_EXPORTS[name] = {"files": paths, "sources": sources, "meaning": meaning}
    plt.show()
    plt.close(fig)
    return paths[0]

pd.set_option("display.max_columns", None)
print("Inputs:", RELEASE.relative_to(root))
print("Outputs:", OUTPUT.relative_to(root))

## Input validation

Check frozen-input hashes, recompute 144 recorded table values, and verify trajectory confidence intervals, warehouse balance and frontier data. Coverage checks separately align 1,100 fine intervals across nine state coordinates. Reference file hashes identify the manuscript assets listed in `docs/paper-assets.csv`.

In [ ]:
validation = validate_inputs()
display(pd.DataFrame([{k: validation[k] for k in ["status", "frozen_files_checked", "table_cells_recomputed"]}]))
assets = pd.read_csv(root / "docs/paper-assets.csv").fillna("")
for asset in assets.itertuples():
    assert hashlib.sha256((root / asset.reference_file).read_bytes()).hexdigest() == asset.reference_sha256
print("PASS: all submission reference outputs match the public asset manifest.")
display(assets[["asset_id", "type", "status", "notes"]])
inventory = pd.read_csv(BASELINE / "inventory_trajectory_by_policy_location.csv")
shipments = pd.read_csv(BASELINE / "weekly_shipments_by_policy_location.csv")
total_shipments = pd.read_csv(BASELINE / "total_shipment_trajectory_by_policy.csv")
coverage, coverage_summary, coverage_validation = load_coverage()
coverage.to_csv(OUTPUT / "data/baseline_r4_e3_aligned_support.csv", index=False)
coverage_summary.to_csv(OUTPUT / "tables/baseline_r4_e3_coverage.csv", index=False)
display(pd.DataFrame([coverage_validation]))

## Weekly demand calibration

The parameter for model week n applies to `(n-1, n]`, including the final interval ending at week 22. Model week 1 corresponds to week 5 of the source data.

In [ ]:
source = "code/configs/fangonda_weekly/theta.csv"
theta = np.loadtxt(SOURCES / source, delimiter=",").reshape(-1)
assert len(theta) == 22
fig, ax = plt.subplots(figsize=STYLE["figsize"], layout="constrained")
ax.stairs(theta, edges=np.arange(23), baseline=None, color=STYLE["colors"][0])
ax.set(xlabel="Week $n$", ylabel=r"$\theta(n)$", xlim=(0,22))
ax.set_xticks([0,4,8,12,16,20,22])
# ax.set_title("Weekly calibration")
save_figure(fig, "theta_calibration_weekly", ["data/sources/"+source], "Weekly parameter on (n-1,n]")

## Baseline fairness frontier

Plot the saved policy evaluations across the baseline fairness weights.

In [ ]:
source = "analysis/baseline_frontier/fairness_weight_frontier_stream16.csv"
frontier = pd.read_csv(SOURCES / source)
ours = frontier[frontier.point == "Our Policy"].sort_values("lambda")
fo = frontier[frontier.point == "Fairness Only"]
fig, ax = plt.subplots(figsize=(6.4,4.3), layout="constrained")
ax.plot(ours.raw_fairness, ours.operating_cost, "-o", color=STYLE["colors"][0], label="Our policy")
ax.scatter(fo.raw_fairness, fo.operating_cost, marker="D", color=STYLE["colors"][1], label="Fairness-only")
for row in ours.to_dict("records"):
    ax.annotate(r"$\lambda=" + f"{row['lambda']:.2f}" + "$", (row["raw_fairness"], row["operating_cost"]), xytext=(5,7), textcoords="offset points", fontsize=9)
ax.set(xlabel="Fairness utility", ylabel="Operating cost")
ax.invert_yaxis()
ax.margins(x=.10, y=.2)
ax.legend(frameon=False, loc="lower left")
save_figure(fig, "fairness_weight_frontier", ["data/sources/"+source], "Raw fairness and operating cost; fixed seed16 policies")

## Health-facility demand frontier

Export the Nh panel separately for the manuscript's combined frontier figure. Historical endpoint qualifications remain recorded in the source data.

In [ ]:
source = "results/common_raw_frontier/common_raw_frontier_extended_with_historical_endpoints.csv"
frontier = pd.read_csv(SOURCES / source)
fig, ax = plt.subplots(figsize=(6.4,4.4), layout="constrained")
for value, color in zip([199, 219, 243], STYLE["colors"]):
    d = frontier[(frontier["Nh"] == value) & (frontier["Q0"] == 142)].sort_values("raw_lambda")
    assert len(d) == 7 and np.allclose(d.raw_lambda, [.1,.15,.25,.35,.5,.7,1.])
    ax.plot(d.fairness_raw, d.operational_cost_raw, "-o", color=color, ms=4, label=r"$N_h=" + f"{value:g}$")
    ax.scatter(d.iloc[0].U_FO, d.iloc[0].C_FO, marker="*", s=130, color=color, edgecolors="white", zorder=5)
ax.set(xlabel="Fairness utility", ylabel="Operating cost")
ax.invert_yaxis()
ax.margins(.07,.12)
legend = ax.legend(frameon=False, loc="upper left")
ax.add_artist(legend)
ax.legend(handles=[Line2D([],[],marker="o",color="#555",label="Our policy"), Line2D([],[],marker="*",ls="",ms=10,color="#555",label="Fairness-only")], frameon=False, loc="lower left")
save_figure(fig, "common_raw_frontier_nh", ["data/sources/"+source], "Separate Nh panel, Q0=142; original endpoint quality flags retained")

## Initial-inventory frontier

Export the Q0 panel separately for the manuscript's combined frontier figure. Historical endpoint qualifications remain recorded in the source data.

In [ ]:
source = "results/common_raw_frontier/common_raw_frontier_extended_with_historical_endpoints.csv"
frontier = pd.read_csv(SOURCES / source)
fig, ax = plt.subplots(figsize=(6.4,4.4), layout="constrained")
for value, color in zip([124.5, 142, 166], STYLE["colors"]):
    d = frontier[(frontier["Q0"] == value) & (frontier["Nh"] == 219)].sort_values("raw_lambda")
    assert len(d) == 7 and np.allclose(d.raw_lambda, [.1,.15,.25,.35,.5,.7,1.])
    ax.plot(d.fairness_raw, d.operational_cost_raw, "-o", color=color, ms=4, label=r"$Q_0=" + f"{value:g}$")
    ax.scatter(d.iloc[0].U_FO, d.iloc[0].C_FO, marker="*", s=130, color=color, edgecolors="white", zorder=5)
ax.set(xlabel="Fairness utility", ylabel="Operating cost")
ax.invert_yaxis()
ax.margins(.07,.12)
legend = ax.legend(frameon=False, loc="upper left")
ax.add_artist(legend)
ax.legend(handles=[Line2D([],[],marker="o",color="#555",label="Our policy"), Line2D([],[],marker="*",ls="",ms=10,color="#555",label="Fairness-only")], frameon=False, loc="lower right")
save_figure(fig, "common_raw_frontier_q0", ["data/sources/"+source], "Separate Q0 panel, Nh=219; original endpoint quality flags retained")

## Additional diagnostic: inventory means and confidence intervals

The nine panels show Q and Z0 through Z7 at weeks 0 through 22. Q is warehouse inventory after dispatch; Z0 is HF inventory, and Z1 through Z7 are CHA inventories at the end of each week. Step lines connect discrete observations and do not imply constant within-week inventories.

Each policy has 10,000 evaluation paths for the fixed seed-16 model. Shading shows pointwise 95% Monte Carlo confidence intervals for the mean. These differ from the empirical path bands used in the manuscript and do not describe training-seed variation.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(10.8, 7.8), sharex=True, layout="constrained")
locations = [(-1, r"$Q$"), *[(i, rf"$Z_{i}$") for i in range(8)]]
for ax, (location, variable) in zip(axes.flat, locations):
    for policy, color in zip(STYLE["baseline_policies"], STYLE["colors"]):
        d = inventory[(inventory.policy_key == policy) & (inventory.location_index == location)].sort_values("week")
        assert np.array_equal(d.week, np.arange(23))
        x, mean = d.week.to_numpy(), d.inventory_mean.to_numpy()
        lower, upper = d.inventory_ci95_lower.to_numpy(), d.inventory_ci95_upper.to_numpy()
        ax.step(x, mean, where="post", marker="o", ms=2.1, color=color, label=POLICIES[policy])
        ax.fill_between(x, lower, upper, step="post", alpha=STYLE["ci_alpha"], color=color, linewidth=0)
        ax.errorbar(x[-1], mean[-1], yerr=[[mean[-1]-lower[-1]], [upper[-1]-mean[-1]]], fmt="none", color=color, capsize=2)
    ax.set(xlabel=r"$n$", ylabel=variable, xlim=(-.15,22.3))
    ax.set_xticks([0,5,10,15,20,22])
    ax.tick_params(axis="x", labelbottom=True)
# Example: axes[0, 0].set_ylim(0, 150)
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside lower center", ncol=2, frameon=False)
save_figure(fig, "baseline_inventory_mean_ci95_grid", [str((BASELINE/"inventory_trajectory_by_policy_location.csv").relative_to(RELEASE))], "3x3 end-of-week inventory mean and pointwise 95% CI; n is week index, step connectors are a display convention")

## Additional diagnostic: shipment means and confidence intervals

The nine panels show total shipment and shipments to the HF and seven CHAs. Shipment for week n applies to `(n-1, n]`, with boundaries 0 through 22. Means and confidence intervals use the same interval boundaries.

Total-shipment standard error is calculated from the total rather than by adding location-specific standard errors. The manuscript shipment figure uses the path bands generated later in this notebook.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(10.8, 7.8), sharex=True, layout="constrained")
series = [(r"$e\cdot a$", total_shipments, "mean_total_shipment", "se_total_shipment")]
series += [(rf"$a_{i}$", shipments[shipments.location_index == i], "mean_shipment", "se_shipment") for i in range(8)]
for ax, (variable, data, mean_column, se_column) in zip(axes.flat, series):
    for policy, color in zip(STYLE["baseline_policies"], STYLE["colors"]):
        d = data[data.policy == policy].sort_values("week")
        assert np.array_equal(d.week, np.arange(1,23))
        mean, se = d[mean_column].to_numpy(), d[se_column].to_numpy()
        lower, upper = mean - 1.96*se, mean + 1.96*se
        edges = np.arange(23)
        ax.stairs(mean, edges=edges, baseline=None, color=color, label=POLICIES[policy])
        ax.fill_between(edges, np.r_[lower,lower[-1]], np.r_[upper,upper[-1]], step="post", alpha=STYLE["ci_alpha"], color=color, linewidth=0)
    ax.set(xlabel=r"$n$", ylabel=variable, xlim=(0,22))
    ax.set_xticks([0,5,10,15,20,22])
    ax.tick_params(axis="x", labelbottom=True)
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside lower center", ncol=2, frameon=False)
save_figure(fig, "baseline_shipments_mean_ci95_grid", [str((BASELINE/f).relative_to(RELEASE)) for f in ["total_shipment_trajectory_by_policy.csv", "weekly_shipments_by_policy_location.csv"]], "3x3 weekly shipment mean +/- 1.96 SE; n-th week shown on (n-1,n], e dot a is total allocation")

## Performance and baseline-parameter tables

Export seven table source views as CSV and LaTeX from the same frozen inputs. The two-stage training table is generated in the next section, giving eight manuscript table mappings in total.

The exported source views may differ from the manuscript tables in row layout and wording. Travel unmet-demand values retain their recorded two-decimal precision. Week-8 quantities and percentages are separate columns. `STYLE` controls display precision and table font size.

In [ ]:
paper_tables = tables.export_tables(STYLE, output=OUTPUT/"tables")
for label, frame in paper_tables.items():
    display(Markdown("### " + label))
    display(frame.style.hide(axis="index").set_properties(**{"font-size": f"{STYLE['table_font_size_pt']}pt"}))
display(read("provenance/audited_travel_precision_audit.csv"))

## Two-stage training details

The table uses parameter names as rows, with columns for M110 and M1100. Values come from the baseline coarse and fine configurations. The output `appendix_e_two_stage_training.csv` corresponds to manuscript label `tab:two_stage_training`; the filename is retained for compatibility.

Historical baseline code snapshots document block mapping, Adam, early stopping, checkpoint restoration and the initial-state mixture. Configuration comparisons cover all 41 coarse and 41 fine dependencies.

Zero-based coarse interval j initializes fine interval `10j+9`, followed by backward training within the block. Each fine interval is trained with a new optimizer, scheduler and early-stopping state. The terminal interval uses the final schedule with early stopping disabled; other block entries use the inner schedule.

Warm starts within a block require adjacent mu, Gamma and eta to agree within tolerance. When parameters change and no coarse initialization is available, the general implementation falls back to the final schedule.

The baseline uses coarse seed 14 and fine seed 16. The run records retain the actual seeds of all initialization dependencies.

In [ ]:
coarse_path = RELEASE / "provenance/run_metadata/55f79fd9206ccca8/config_used.json"
fine_path = RELEASE / "provenance/run_metadata/bd3799989041ba15/config_used.json"
coarse, fine = [json.loads(p.read_text()) for p in [coarse_path, fine_path]]
resolved = json.loads((fine_path.parent/"provenance/resolved_configuration.json").read_text())
assert resolved["initialization"]["block_size"] == 10
assert resolved["scientific"]["primary_extraction"] == "E3"
assert not any(resolved["initialization"][k] for k in ["optimizer_state_reused", "scheduler_state_reused", "early_stopping_state_reused"])
historical_manifest = json.loads((RELEASE/"implementation_audit/historical_code/source_manifest.json").read_text())
for item in historical_manifest:
    assert hashlib.sha256((RELEASE/item["bundle_path"]).read_bytes()).hexdigest() == item["sha256"]
# All row values / wording are editable here; config-derived fields remain traceable.
cn, fn = coarse["net_config"], fine["net_config"]
ce, fe = coarse["eqn_config"], fine["eqn_config"]
rows = []
def training_row(parameter, m110, m1100=None):
    rows.append({"Parameter": parameter, "M110 — coarse": str(m110), "M1100 — fine": str(m110 if m1100 is None else m1100)})
training_row("Time horizon (weeks)", ce["total_time"], fe["total_time"])
training_row("Time intervals / steps per week", "110 / 5", "1100 / 50")
training_row("Time step (weeks)", "0.2", "0.02")
training_row("State coordinates", "Q, Z0, ..., Z7 (9 coordinates)")
training_row("Hidden layers × width", f"{cn['num_layers']} × {cn['num_neurons']}", f"{fn['num_layers']} × {fn['num_neurons']}")
training_row("Activation", cn["activation"], fn["activation"])
training_row("Fresh H / G initialization", f"{cn['h_init_method']} / {cn['g_init_method']}")
training_row("G output bias", cn["g_output_bias"], fn["g_output_bias"])
training_row("Input normalization", "State divided by scenario Q0 (baseline 142)")
training_row("H output scaling", "H = H_hat / " + str(cn["h_output_scale"]))
training_row("Loss", cn["loss_mode"], fn["loss_mode"])
training_row("Consistency coefficient", cn["consistency_coef"], fn["consistency_coef"])
training_row("Optimizer", "Adam, joint H/G parameters; default betas/epsilon")
training_row("Batch size", cn["batch_size"], fn["batch_size"])
training_row("Validation size per interval", cn["valid_size"], fn["valid_size"])
training_row("Terminal learning rate", cn["final_learning_rate"], fn["final_learning_rate"])
training_row("Terminal iteration budget", cn["final_iterations"], fn["final_iterations"])
training_row("Terminal LR milestones", cn["final_milestones"], fn["final_milestones"])
training_row("Interior learning rate", cn["inner_learning_rate"], fn["inner_learning_rate"])
training_row("Interior maximum iterations", cn["inner_iterations"], fn["inner_iterations"])
training_row("Interior LR milestones", cn["inner_milestones"], fn["inner_milestones"])
training_row("LR multiplier (terminal / interior)", f"{cn['final_gamma']} / {cn['inner_gamma']}")
training_row("Validation every (iterations)", cn["validation_every"], fn["validation_every"])
training_row("Early stopping: terminal / interior", "Disabled / enabled")
training_row("Interior minimum iterations", cn["inner_min_iterations"], fn["inner_min_iterations"])
training_row("Patience setting (iterations)", cn["patience"], fn["patience"])
training_row("Stop after non-improving checks", "> 60 (implementation uses strict >)", "> 30 (implementation uses strict >)")
training_row("Minimum improvement", cn["min_delta"], fn["min_delta"])
training_row("Checkpoint retained", "Lowest validation loss checkpoint restored")
training_row("Gradient clipping", "Joint H/G norm, maximum " + str(cn["grad_norm"]))
training_row("Reference mixture", ce["reference_mixture"], fe["reference_mixture"])
training_row("R0 / R1 / R2 rollout counts", ce["reference_mixture_counts"], fe["reference_mixture_counts"])
training_row("Rollout pool size", cn["rollout_pool_size"], fn["rollout_pool_size"])
training_row("Sampler weights: rollout / box / shortage", "1 / 0 / 0")
training_row("Initial-state mixture", "Nominal with probability 0.7; perturbed with probability 0.3")
training_row("Perturbed initial Q", "Uniform [0.95 Q0, 1.05 Q0]")
training_row("Perturbed initial Zi", "Independent uniform [0, 0.0125 Q0]")
training_row("Training order", "Backward over all intervals")
training_row("Block-entry initialization", "Fresh terminal H/G; then backward warm start", "Coarse j initializes fine 10j+9; block size 10")
training_row("Adjacent-interval warm start", "Unconditional", "Only if mu, Gamma, eta match; tolerance 1e-8")
training_row("Coarse optimizer state inherited", "Not applicable", "No optimizer / scheduler / early-stop state inherited")
training_row("Optimizer / scheduler per interval", "Reinitialized")
training_details = pd.DataFrame(rows).set_index("Parameter")
display(training_details.style.set_properties(**{"font-size": f"{STYLE['table_font_size_pt']}pt", "white-space": "normal"}))
training_details.to_csv(OUTPUT/"tables/appendix_e_two_stage_training.csv")
# Longtable avoids shrinking a detailed table to unreadably small type. Requires longtable/booktabs.
training_details.to_latex(OUTPUT/"tables/appendix_e_two_stage_training.tex", index=True, escape=True, longtable=True,
    column_format=r"p{0.28\textwidth}p{0.30\textwidth}p{0.30\textwidth}",
    caption="Two-stage training settings (baseline configuration; scenario-specific Q0).", label="tab:two_stage_training")
training_provenance = {"coarse_config": str(coarse_path.relative_to(RELEASE)), "fine_config": str(fine_path.relative_to(RELEASE)),
    "coarse_seed": cn["train_seed"], "fine_seed": fn["train_seed"], "historical_code": historical_manifest,
    "scope": "Baseline implementation and settings; shared fields checked across the 41 coarse and 41 fine configurations used in the paper."}
(OUTPUT/"data/training_table_provenance.json").write_text(json.dumps(training_provenance, indent=2))

### Configuration checks and scope

Compare the fields in the training table across all selected configurations. Inventory normalization uses each scenario's Q0. Implementation-level checks cover the two baseline historical commits; they do not establish identical behavior for every historical version.

Q0 comparisons allow an absolute tolerance of `1e-8`. Some recorded I0 values contain floating-point differences, such as `124.50000000004678` and `166.0000000000624`, for scales 124.5 and 166. The original values remain in the configurations and audit outputs.

In [ ]:
run_index = read("implementation_audit/run_index.csv")
config_checks = []
net_fields = ["num_layers", "num_neurons", "activation", "h_init_method", "g_init_method", "g_output_bias", "input_mode", "h_output_scale", "loss_mode", "consistency_coef", "batch_size", "valid_size", "final_learning_rate", "final_iterations", "final_milestones", "inner_learning_rate", "inner_iterations", "inner_milestones", "final_gamma", "inner_gamma", "validation_every", "early_stopping", "inner_min_iterations", "patience", "min_delta", "grad_norm", "grad_clip_rule", "sampling_mode", "mix_rollout", "mix_box", "mix_shortage", "rollout_pool_size", "initial_state_mode", "initial_perturb_prob", "initial_q_perturb_frac", "initial_z_perturb_frac", "prev_init", "prev_init_same_params_only", "prev_init_param_tol"]
for run in run_index.itertuples():
    config = json.loads((RELEASE/run.config_path).read_text())
    base = coarse if run.M == 110 else fine
    for field in net_fields:
        actual, expected = config["net_config"].get(field), base["net_config"].get(field)
        config_checks.append({"run": run.run_dir, "M": run.M, "field": "net_config."+field, "actual": json.dumps(actual), "expected": json.dumps(expected), "matches": actual == expected})
    actual, expected = config["net_config"]["q0_input_scale"], config["eqn_config"]["I0"]
    config_checks.append({"run": run.run_dir, "M": run.M, "field": "q0_input_scale ~= scenario I0 (atol 1e-8)", "actual": actual, "expected": expected, "matches": bool(np.isclose(actual, expected, rtol=0, atol=1e-8))})
config_checks = pd.DataFrame(config_checks)
config_checks.to_csv(OUTPUT/"data/training_table_config_checks.csv", index=False)
display(config_checks.groupby("M").agg(checks=("matches","size"), matching=("matches","sum")))
display(config_checks[~config_checks.matches])


## Reference-state coverage

For the baseline model (lambda 0.25, M1100, seed 16), compare Q and Z0 through Z7 separately. **Reference** denotes the `training_sampler` states: 16,384 logged validation states per fine interval from a 32,768-path R4 rollout pool. The separately recorded `reference_rollout` sample is not used for this comparison.

**Policy** denotes `discrete_review_deep_splitting` virtual E3 network query states, with 10,000 queries per interval. These differ from fine-grid policy evaluation states and physical inventories at review-period ends.

Fine intervals align by `t = interval × 22 / 1100`, ending at t=21.98 weeks. Shading shows the marginal 1–99% quantile bands and the recorded min–max ranges; lines show medians. There is no terminal query at t=22.

Outside-range fractions use the original path-level logs, preserving their thresholds and numerical precision. Marginal quantiles alone cannot recover those fractions. The comparison measures empirical marginal coverage, not nine-dimensional joint coverage or coverage of an unknown optimal policy.

The manuscript figure places the nine coordinates in a 3×3 grid. Legend labels are defined in `COVERAGE_LABELS`.

In [ ]:
COVERAGE_LABELS = {
    "reference_range": "Reference policy: range",
    "reference_band": "Reference policy: 1–99%",
    "reference_median": "Reference policy: median",
    "policy_band": "Our policy: 1–99%",
    "policy_median": "Our policy: median",
    "outside_range": "Outside range",
    "outside_band": "Outside 1–99%",
}
fig, axes = plt.subplots(3, 3, figsize=(10.8, 8.0), sharex=True, layout="constrained")
variables = [("Q", r"$Q$"), *[(f"Z_{i}", rf"$Z_{i}$") for i in range(8)]]
for ax, (key, variable) in zip(axes.flat, variables):
    d = coverage[coverage.variable == key].sort_values("interval")
    x = d.time_weeks.to_numpy()
    ax.fill_between(x, d.min_r4, d.max_r4, color=STYLE["colors"][0], alpha=.10, step="post", linewidth=0, label=COVERAGE_LABELS["reference_range"])
    ax.fill_between(x, d.p01_r4, d.p99_r4, color=STYLE["colors"][0], alpha=.26, step="post", linewidth=0, label=COVERAGE_LABELS["reference_band"])
    ax.step(x, d.p50_r4, where="post", color=STYLE["colors"][0], lw=1.3, label=COVERAGE_LABELS["reference_median"])
    ax.fill_between(x, d.p01_e3, d.p99_e3, color=STYLE["colors"][1], alpha=.16, step="post", linewidth=0, label=COVERAGE_LABELS["policy_band"])
    ax.step(x, d.p50_e3, where="post", color=STYLE["colors"][1], ls="--", lw=1.3, label=COVERAGE_LABELS["policy_median"])
    ax.set(xlabel=r"$n$", ylabel=variable, xlim=(0,22))
    ax.set_xticks([0,5,10,15,20,22])
    ax.tick_params(axis="x", labelbottom=True)
handles, labels = axes.flat[0].get_legend_handles_labels()
# Arrange three Reference entries followed by two Policy entries; legends fill by column.
order = [0,3,1,4,2]
fig.legend([handles[i] for i in order], [labels[i] for i in order], loc="outside lower center", ncol=3, frameon=False, fontsize=9)
save_figure(fig, "baseline_reference_policy_support_grid", ["data/baseline_coverage/state_support_summary.csv", "data/baseline_coverage/evaluation_state_support_summary.csv"], "3x3 logged R4 reference marginal support versus E3 virtual policy queries; display labels Reference and Policy; quantile bands, not mean CIs")

## Additional diagnostic: outside-range fractions

Average the recorded fractions using query-count weights across fine intervals, each with 10,000 queries. This is not an average of quantile values. Maximum interval-specific fractions are also reported to show local exceptions.

Queries along a path are correlated, so the 1,100 × 10,000 observations are not treated as independent samples for a confidence interval. Zero recorded exceedances describe these samples only. "Outside range" and "Outside 1–99%" refer to the reference min–max range and reference quantile band, respectively.

In [ ]:
view = coverage_summary[["variable", "outside_min_max_pct", "outside_p01_p99_pct", "peak_outside_min_max_pct", "peak_outside_p01_p99_pct"]].copy()
view.columns = ["Coordinate", "Outside min–max (%)", "Outside 1–99% (%)", "Peak outside min–max (%)", "Peak outside 1–99% (%)"]
display(view.style.hide(axis="index").format({c: "{:.4f}" for c in view.columns[1:]}))
view.to_latex(OUTPUT/"tables/baseline_r4_e3_coverage.tex", index=False, float_format="%.4f", escape=True,
    caption="Baseline marginal E3 query-state exceedance relative to the logged R4 sample.", label="tab:r4_e3_coverage")
fig, ax = plt.subplots(figsize=(7.2,3.8), layout="constrained")
x = np.arange(len(coverage_summary))
ax.bar(x-.18, coverage_summary.outside_min_max_pct, width=.36, color=STYLE["colors"][0], label=COVERAGE_LABELS["outside_range"])
ax.bar(x+.18, coverage_summary.outside_p01_p99_pct, width=.36, color=STYLE["colors"][1], label=COVERAGE_LABELS["outside_band"])
ax.set_xticks(x, [v.replace("_", "") for v in coverage_summary.variable])
ax.set(xlabel="State coordinate", ylabel="Query-count-weighted outside fraction (%)")
ax.legend(frameon=False)
ax.margins(y=.18)
save_figure(fig, "baseline_r4_e3_outside_fraction", ["data/baseline_coverage/evaluation_state_support_summary.csv"], "Count-weighted marginal exceedance over 1100 fine intervals; no independent-sample CI")

## Additional diagnostic: Q and HF coverage over time

Plot fine-interval outside-range fractions for Q and HF inventory. The remaining CHA coordinates have no recorded exceedances in these logs; the exported aligned-support CSV contains all coordinates. These time series describe network query states, not weekly inventory trajectories.

### Warehouse inventory (Q)

In [ ]:
d = coverage[coverage.variable == "Q"].sort_values("interval")
fig, ax = plt.subplots(figsize=STYLE["figsize"], layout="constrained")
ax.step(d.time_weeks, 100*d.outside_training_min_max_fraction, where="post", color=STYLE["colors"][0], label=COVERAGE_LABELS["outside_range"])
ax.step(d.time_weeks, 100*d.outside_training_p01_p99_fraction, where="post", color=STYLE["colors"][1], label=COVERAGE_LABELS["outside_band"])
ax.set(xlabel="$t$", ylabel=r"$p_{\mathrm{out},Q}$ (%)", xlim=(0,22))
ax.set_xticks([0,4,8,12,16,20,22])
ax.legend(frameon=False)
save_figure(fig, "baseline_r4_e3_outside_time_q", ["data/baseline_coverage/evaluation_state_support_summary.csv"], "Recorded fine-interval marginal out-of-range fractions")

### Health-facility inventory (Z0)

In [ ]:
d = coverage[coverage.variable == "Z_0"].sort_values("interval")
fig, ax = plt.subplots(figsize=STYLE["figsize"], layout="constrained")
ax.step(d.time_weeks, 100*d.outside_training_min_max_fraction, where="post", color=STYLE["colors"][0], label=COVERAGE_LABELS["outside_range"])
ax.step(d.time_weeks, 100*d.outside_training_p01_p99_fraction, where="post", color=STYLE["colors"][1], label=COVERAGE_LABELS["outside_band"])
ax.set(xlabel="$t$", ylabel=r"$p_{\mathrm{out},Z_0}$ (%)", xlim=(0,22))
ax.set_xticks([0,4,8,12,16,20,22])
ax.legend(frameon=False)
save_figure(fig, "baseline_r4_e3_outside_time_z_0", ["data/baseline_coverage/evaluation_state_support_summary.csv"], "Recorded fine-interval marginal out-of-range fractions")

### A local boundary case for Q

At t=0.98 weeks, all E3 queries for Q equal 142, while the recorded reference maximum is approximately 141.9953. The outside-range fraction is therefore 100%, with an exceedance of about 0.004745 inventory units.

The table retains the original threshold, fraction and magnitude. This local event should be read alongside the overall mean and peak exceedance statistics; it alone does not establish policy unreliability.

In [ ]:
q_peak = coverage[(coverage.variable == "Q") & (coverage.interval == 49)].copy()
q_peak["max_exceedance_units"] = np.maximum(q_peak.max_e3 - q_peak.max_r4, 0)
peak_columns = ["interval", "time_weeks", "min_r4", "max_r4", "min_e3", "max_e3", "outside_training_min_max_fraction", "max_exceedance_units", "max_normalized_distance_outside_training_min_max"]
display(q_peak[peak_columns].T)
q_peak[peak_columns].to_csv(OUTPUT/"tables/baseline_q_boundary_diagnostic.csv", index=False)

## Output index and validation records

`figure_manifest.json` lists the figures generated by the preceding sections. The subsequent trajectory section writes its path-band outputs separately. Use `docs/paper-assets.csv` for the complete mapping to manuscript figures and tables.

Training checks cover the 82 reported model configurations. Coverage checks describe empirical marginal support for the baseline policy, not joint support or the unknown optimal policy. Table exports provide numerical source views; manuscript table environments are available in `reference_outputs/tables/`.

In [ ]:
import importlib.metadata as metadata
(OUTPUT/"figure_manifest.json").write_text(json.dumps(FIGURE_EXPORTS, ensure_ascii=False, indent=2))
(OUTPUT/"style_used.json").write_text(json.dumps(STYLE, ensure_ascii=False, indent=2))
environment = {name: metadata.version(name) for name in ["numpy", "pandas", "matplotlib", "nbformat", "nbclient", "ipykernel"]}
environment.update(python=sys.version, interpreter="active Python environment")
(OUTPUT/"environment.json").write_text(json.dumps(environment, indent=2))
report = {"inputs": validation, "coverage": coverage_validation, "training_config_checks": len(config_checks),
          "training_config_mismatches": int((~config_checks.matches).sum()), "figure_count": len(FIGURE_EXPORTS),
          "manuscript_tables": len(paper_tables), "training_detail_rows": len(training_details)}
(OUTPUT/"validation.json").write_text(json.dumps(report, ensure_ascii=False, indent=2))
display(pd.DataFrame([{"figure": k, "pdf": v["files"][0]} for k,v in FIGURE_EXPORTS.items()]))

print(f"Generated {len(FIGURE_EXPORTS)} PDF/PNG figure sets; seven table source views, the training details table and coverage summaries. Path-band figures are generated below.")
print(OUTPUT.relative_to(root))

## Inventory and shipments: manuscript path-band figures

This section uses the saved 10,000 periodic-review evaluation paths for the baseline seed-16 model. The **5–95% path-band figures** are the manuscript inventory and shipment outputs. The 1–99% variants are additional diagnostics. Files are saved under `exports/trajectory_distributions/`.

Blue lines show the mean of Our policy; shading shows pointwise empirical path quantiles. These bands are neither confidence intervals for the mean nor simultaneous bands for entire trajectories. Orange lines show calibrated weekly local mean demand. HF demand excludes referrals; warehouse demand denotes the system total across the HF and seven CHAs.

Inventory at n is an end-of-week observation; demand and shipment for week n apply to `(n-1, n]`. Shipment shares an axis with demand, while inventory panels place demand on the right axis.

Z and local unmet demand are read from the saved paths. Shipments are reconstructed from inventory balance, and Q from cumulative shipments. Reconstructed means and standard errors are checked against saved summaries. Small negative Q values caused by floating-point error, of magnitude approximately 3e-5 or less, are clipped to zero for display; raw values remain in the audit output.

`BANDS` and the plotting parameters control quantiles, colors, fonts and panel sizes.

In [ ]:
from pathlib import Path
import sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "code/paper_analysis").is_dir())
if str(root / "code") not in sys.path:
    sys.path.insert(0, str(root / "code"))
from paper_analysis.data import RELEASE, BASELINE, SOURCES
PATH_DATA = RELEASE / "data/baseline_path_variability_v1"
PREVIEW = RELEASE / "exports/trajectory_distributions"
PREVIEW.mkdir(parents=True, exist_ok=True)
manifest = json.loads((PATH_DATA / "extended_evaluation_manifest_v1.json").read_text())
bank = json.loads((PATH_DATA / "evaluation_bank_used.json").read_text())
assert manifest["metadata"]["evaluation_bank_sha256"] == bank["manifest_sha256"]
assert manifest["metadata"]["setting_id"] == "nominal_lam0p25"
assert manifest["metadata"]["stream_id"] == 16
hashes = {"periodic_review_state_histories_v1.npz": next(x["sha256"] for x in manifest["artifacts"] if x["path"] == "periodic_review_state_histories_v1.npz"),
          "periodic_review_demands.npy": bank["arrays"]["periodic_review_demands"]["file_sha256"]}
for filename, expected in hashes.items():
    assert hashlib.sha256((PATH_DATA / filename).read_bytes()).hexdigest() == expected
with np.load(PATH_DATA / "periodic_review_state_histories_v1.npz", allow_pickle=False) as archive:
    z_paths = archive["E3_mean_rollout__z_hist"].astype(np.float64)
    unmet_paths = archive["E3_mean_rollout__l_hist"].astype(np.float64)
demand_paths = np.load(PATH_DATA / "periodic_review_demands.npy", allow_pickle=False).astype(np.float64)
assert z_paths.shape == (23, 10000, 8)
assert unmet_paths.shape == demand_paths.shape == (22, 10000, 8)
params = SOURCES / "code/configs/fangonda_weekly"
mu = np.loadtxt(params / "mu.csv", delimiter=",")
p_referral = np.loadtxt(params / "p.csv", delimiter=",").reshape(7)
assert mu.shape == (22, 8)
# Z(n) = Z(n-1) + a(n) - effective demand(n) + local unmet(n).
effective_demand = demand_paths.copy()
effective_demand[:, :, 0] += (unmet_paths[:, :, 1:] * p_referral).sum(axis=2)
a_paths = np.diff(z_paths, axis=0) + effective_demand - unmet_paths
q_paths = np.concatenate([np.full((1, 10000), 142.), 142. - np.cumsum(a_paths.sum(axis=2), axis=0)])
assert np.isfinite(a_paths).all() and np.isfinite(q_paths).all()
assert a_paths.min() >= -1e-5 and q_paths.min() >= -5e-5
inventory_ref = pd.read_csv(BASELINE / "inventory_trajectory_by_policy_location.csv")
shipment_ref = pd.read_csv(BASELINE / "weekly_shipments_by_policy_location.csv")
total_ref = pd.read_csv(BASELINE / "total_shipment_trajectory_by_policy.csv")
checks = []
for location in range(-1, 8):
    x = q_paths if location == -1 else z_paths[:, :, location]
    ref = inventory_ref[(inventory_ref.policy_key == "E3_mean_rollout") & (inventory_ref.location_index == location)].sort_values("week")
    for field, value in [("mean", x.mean(1)), ("se", x.std(1, ddof=1)/100)]:
        delta = float(np.max(np.abs(value-ref["inventory_"+field].to_numpy())))
        assert delta < 1e-5, (location, field, delta)
        checks.append({"kind": "inventory", "location": location, "stat": field, "max_abs_error": delta})
    x = a_paths.sum(2) if location == -1 else a_paths[:, :, location]
    ref = total_ref[total_ref.policy == "E3_mean_rollout"].sort_values("week") if location == -1 else shipment_ref[(shipment_ref.policy == "E3_mean_rollout") & (shipment_ref.location_index == location)].sort_values("week")
    for field, value in [("mean", x.mean(1)), ("se", x.std(1, ddof=1)/100)]:
        column = (field+"_total_shipment") if location == -1 else (field+"_shipment")
        delta = float(np.max(np.abs(value-ref[column].to_numpy())))
        assert delta < 1e-5, (location, field, delta)
        checks.append({"kind": "shipment", "location": location, "stat": field, "max_abs_error": delta})
pd.DataFrame(checks).to_csv(PREVIEW / "reconstruction_checks.csv", index=False)
quantile_rows = []
for kind in ["inventory", "shipment"]:
    for location in range(-1, 8):
        x = (q_paths if location == -1 else z_paths[:, :, location]) if kind == "inventory" else (a_paths.sum(2) if location == -1 else a_paths[:, :, location])
        quantiles = np.quantile(x, [.01, .05, .5, .95, .99], axis=1)
        for j in range(len(x)):
            quantile_rows.append(dict(kind=kind, location=location, week=j+(kind=="shipment"), mean=x[j].mean(), **dict(zip(["p01","p05","p50","p95","p99"], quantiles[:,j]))))
pd.DataFrame(quantile_rows).to_csv(PREVIEW / "path_quantiles.csv", index=False)
report = dict(status="PASS", paths=10000, hashes=hashes, q_reconstructed_min=float(q_paths.min()), checks=len(checks),
              reconstruction="a=diff(Z)+native demand+HF referrals-local unmet; Q=142-cumulative total a",
              source_kind="saved_simulation_results", source_artifact="external/artifact-389b5d18db41c2ad",
              demand="calibrated native weekly means; HF excludes referrals; Warehouse uses system sum",
              bands="pointwise empirical path quantiles, not confidence intervals",
              q_display="clip negative roundoff to zero; retain raw values in path_quantiles.csv")
(PREVIEW / "validation.json").write_text(json.dumps(report, indent=2))
print("PASS: historical paths verified; 36 mean/SE comparisons match within 1e-5.")


In [ ]:
# Plot settings: quantile bands, colors, font sizes and figure dimensions.
BANDS = [(0.01, 0.99), (0.05, 0.95)]
BLUE, ORANGE = "#0072B2", "#D55E00"
preview_files = []
with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 10,
                     "axes.labelsize": 10, "axes.titlesize": 12,
                     "xtick.labelsize": 9, "ytick.labelsize": 9,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "pdf.fonttype": 42, "figure.dpi": 110}):
    for lower_p, upper_p in BANDS:
        band_label = f"{100*lower_p:.0f}–{100*upper_p:.0f}% of paths (pointwise)"
        for kind in ["shipment", "inventory"]:
            fig, axes = plt.subplots(3, 3, figsize=(12.4, 9.2), layout="constrained")
            for ax, location in zip(axes.flat, range(-1, 8)):
                title = "Warehouse" if location == -1 else ("HF" if location == 0 else f"CHA{location}")
                demand = mu.sum(1) if location == -1 else mu[:, location]
                if kind == "shipment":
                    values = a_paths.sum(2) if location == -1 else a_paths[:, :, location]
                    mean = values.mean(1)
                    lo, hi = np.quantile(values, [lower_p, upper_p], axis=1)
                    edges = np.arange(23)
                    ax.fill_between(edges, np.r_[lo,lo[-1]], np.r_[hi,hi[-1]], step="post", color=BLUE, alpha=.22, linewidth=0)
                    ax.stairs(mean, edges, baseline=None, color=BLUE, lw=1.7)
                    ax.set_ylabel(r"$e\cdot a(n)$" if location == -1 else rf"$a_{location}(n)$")
                    demand_ax = ax
                else:
                    values = q_paths if location == -1 else z_paths[:, :, location]
                    mean = np.maximum(values.mean(1), 0.)
                    lo, hi = np.maximum(np.quantile(values, [lower_p, upper_p], axis=1), 0.)
                    ax.fill_between(np.arange(23), lo, hi, step="post", color=BLUE, alpha=.22, linewidth=0)
                    ax.step(np.arange(23), mean, where="post", color=BLUE, lw=1.7)
                    # Show the final quantile range at week 22 as a vertical line.
                    ax.vlines(22, lo[-1], hi[-1], color=BLUE, alpha=.45, lw=2)
                    ax.set_ylabel(r"$Q(n)$" if location == -1 else rf"$Z_{location}(n)$", color=BLUE)
                    ax.tick_params(axis="y", colors=BLUE)
                    demand_ax = ax.twinx()
                    demand_ax.spines["right"].set_visible(True)
                    demand_ax.spines["right"].set_color(ORANGE)
                    demand_ax.tick_params(axis="y", colors=ORANGE)
                    demand_ax.set_ylabel("Mean demand / week", color=ORANGE, fontsize=9)
                    demand_ax.set_ylim(0, float(demand.max())*1.30)
                demand_ax.stairs(demand, np.arange(23), baseline=None, color=ORANGE, ls="--", lw=1.4)
                ax.set(title=title, xlabel=r"$n$", xlim=(-.05,22.25) if kind=="inventory" else (0,22))
                ax.set_xticks([0,5,10,15,20,22])
                ax.set_ylim(bottom=0)
                ax.margins(y=.12)
                ax.grid(axis="y", color="#cccccc", alpha=.3, lw=.5)
            handles = [Line2D([],[],color=BLUE,lw=1.7,label="Our policy: mean"),
                       Patch(facecolor=BLUE,alpha=.22,label=band_label),
                       Line2D([],[],color=ORANGE,lw=1.4,ls="--",label="Mean demand" + (" (right axis)" if kind=="inventory" else ""))]
            fig.legend(handles=handles, loc="outside lower center", ncol=3, frameon=False, fontsize=10)
            name = f"baseline_{kind}_demand_p{int(lower_p*100):02d}_p{int(upper_p*100):02d}"
            fig.canvas.draw()
            fig.set_layout_engine("none")
            for ext in ["pdf","png"]:
                file = PREVIEW / f"{name}.{ext}"
                fig.savefig(file, dpi=180, bbox_inches="tight", facecolor="white")
                preview_files.append(file.name)
            plt.show()
            plt.close(fig)
(PREVIEW / "figure_manifest.json").write_text(json.dumps({"files":preview_files, "validation":"validation.json", "data":"path_quantiles.csv"},indent=2))
print(PREVIEW)


## Additional diagnostic: weekly inventory distributions

Each column is a histogram of the 10,000 inventory observations at week n, using 50 equal-width inventory bins. Color represents the fraction of paths in a bin on a logarithmic scale. Bin edges differ by location, so the colors represent bin probabilities rather than continuous probability densities.

The full observed range is retained, including the mass at zero in the lowest bin. Black lines show inventory means; orange lines show local mean demand on the right axis. Columns are centered on discrete observation times and do not represent continuous within-week dynamics.

Only the heatmap layer is rasterized in PDF output; text and lines remain vector elements. Run the preceding path-loading section before this cell.

In [ ]:
from matplotlib.colors import LogNorm
HEATMAP_BINS = 50
HEATMAP_NORM = LogNorm(vmin=1/10000, vmax=1)
BLUE, ORANGE = "#0072B2", "#D55E00"
with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 10,
                     "axes.labelsize": 10, "axes.titlesize": 12,
                     "xtick.labelsize": 9, "ytick.labelsize": 9,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "pdf.fonttype": 42, "figure.dpi": 110}):
    fig, axes = plt.subplots(3, 3, figsize=(12.8, 9.2), layout="constrained")
    heatmap_checks = []
    for ax, location in zip(axes.flat, range(-1,8)):
        values = np.maximum(q_paths if location == -1 else z_paths[:, :, location],0.)
        upper = max(float(values.max())*1.001, .01)
        yedges = np.linspace(0, upper, HEATMAP_BINS+1)
        probabilities = np.stack([np.histogram(v,bins=yedges)[0]/len(v) for v in values],axis=1)
        assert np.allclose(probabilities.sum(axis=0),1.)
        heatmap_checks.append(dict(location=location, bins=HEATMAP_BINS, upper=upper,
                                   min_column_sum=float(probabilities.sum(axis=0).min()),
                                   max_column_sum=float(probabilities.sum(axis=0).max())))
        mesh = ax.pcolormesh(np.arange(24)-.5, yedges, np.ma.masked_equal(probabilities,0),
                             cmap="Blues", norm=HEATMAP_NORM, shading="flat", rasterized=True)
        ax.step(np.arange(23),values.mean(1),where="post",color="#202020",lw=1.7)
        demand = mu.sum(1) if location == -1 else mu[:,location]
        demand_ax=ax.twinx()
        demand_ax.spines["right"].set_visible(True)
        demand_ax.spines["right"].set_color(ORANGE)
        demand_ax.tick_params(axis="y",colors=ORANGE)
        demand_ax.set_ylabel("Mean demand / week",color=ORANGE,fontsize=9)
        demand_ax.stairs(demand,np.arange(23),baseline=None,color=ORANGE,ls="--",lw=1.4)
        demand_ax.set_ylim(0,float(demand.max())*1.3)
        title="Warehouse" if location == -1 else ("HF" if location == 0 else f"CHA{location}")
        ax.set(title=title,xlabel=r"$n$",xlim=(-.5,22.5),ylim=(0,upper),
               ylabel=r"$Q(n)$" if location == -1 else rf"$Z_{location}(n)$")
        ax.set_xticks([0,5,10,15,20,22])
    colorbar=fig.colorbar(mesh,ax=axes.ravel().tolist(),location="bottom",shrink=.68,pad=.055,aspect=50)
    colorbar.set_ticks([.0001,.001,.01,.1,1.],labels=["0.01%","0.1%","1%","10%","100%"])
    colorbar.set_label("Paths in each inventory bin (log color scale)",fontsize=10)
    colorbar.ax.minorticks_off()
    fig.legend(handles=[Line2D([],[],color="#202020",lw=1.7,label="Our policy: mean"),
                         Line2D([],[],color=ORANGE,ls="--",lw=1.4,label="Mean demand (right axis)")],
               loc="outside lower center",ncol=2,frameon=False,fontsize=10)
    fig.canvas.draw()
    fig.set_layout_engine("none")
    for ext in ["pdf","png"]:
        fig.savefig(PREVIEW/f"baseline_inventory_demand_heatmap.{ext}",dpi=180,bbox_inches="tight",facecolor="white")
    plt.show()
    plt.close(fig)
pd.DataFrame(heatmap_checks).to_csv(PREVIEW/"heatmap_checks.csv",index=False)
print("PASS: every weekly histogram contains all 10,000 paths; no tail truncation.")
